# Imports

In [35]:
import os
from pathlib import Path
import sys
import requests
import pandas as pd
from io import StringIO
project_root = Path.cwd().parent
sys.path.insert(0, str(project_root))
os.chdir(project_root)
from src.data_loader import load_reservoir_data
from src.mongo_loader import load_mongo_data
from src.weather_loader import download_weather
from river import linear_model, preprocessing, optim, metrics
import plotly.graph_objects as go
import json

# Get exchange rates

In [3]:
def get_exchange_rates(start_date, end_date):
    url = "https://data.norges-bank.no/api/data/EXR/B.EUR+USD.NOK.SP"

    params = {
        "startPeriod": start_date,
        "endPeriod": end_date,
        "format": "csv"
    }

    response = requests.get(url, params=params, timeout=30)
    response.raise_for_status()

    df = pd.read_csv(StringIO(response.text), sep=";")

    #Extracting relevant columns

    df = df[["TIME_PERIOD", "BASE_CUR", "OBS_VALUE"]].copy()

    df.columns = ["date", "currency", "exchange_rate"]

    df["date"] = pd.to_datetime(df["date"])
    df["exchange_rate"] = pd.to_numeric(df["exchange_rate"])

    return df

In [4]:
exchange_rates = get_exchange_rates("2026-01-01", "2026-01-31")

display(exchange_rates.head())

,date,currency,exchange_rate
0,2026-01-02,USD,10.0661
1,2026-01-05,USD,10.0990
2,2026-01-06,USD,10.0184
3,2026-01-07,USD,10.0638
4,2026-01-08,USD,10.1015


# Checking timezones


In [6]:
df = load_reservoir_data()
print("NVE:")
print(df["date"].dtype)
print(df["date"].head())

print("Norges Bank:")
print(exchange_rates["date"].dtype)
print(exchange_rates["date"].head())

2026-11-23 10:47:34.509 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.


NVE:
datetime64[us]
14903   1995-01-08
9951    1995-01-08
7718    1995-01-08
11268   1995-01-08
7118    1995-01-08
Name: date, dtype: datetime64[us]
Norges Bank:
datetime64[us]
0   2026-01-02
1   2026-01-05
2   2026-01-06
3   2026-01-07
4   2026-01-08
Name: date, dtype: datetime64[us]


In [7]:
weather_df = download_weather(
    "NO 3",
    "2024-01-01",
    "2024-12-31"
)

#Adding back the UTC timezone to the ENTSOE data.
entsoe_df = load_mongo_data()
entsoe_df["timestamp"] = pd.to_datetime(
    entsoe_df["timestamp"],
    utc=True
)

Loading saved batch 0
Loading saved batch 100
Loading saved batch 200
Loading saved batch 300
Loading saved batch 400
Loading saved batch 500
Loading saved batch 600
Loading saved batch 700
Loading saved batch 800
Loading saved batch 900


2026-11-23 10:48:02.675 WARNING streamlit.runtime.caching.cache_data_api: No runtime found, using MemoryCacheStorageManager
2026-11-23 10:48:02.675 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.


In [8]:
print("Open-meteo:")
print(weather_df["date"].dtype)
print(weather_df["date"].head())

print("ENTSO-E:")
print(entsoe_df["timestamp"].dtype)
print(entsoe_df["timestamp"].head())

Open-meteo:
datetime64[us, UTC]
0   2024-01-01 00:00:00+00:00
1   2024-01-01 01:00:00+00:00
2   2024-01-01 02:00:00+00:00
3   2024-01-01 03:00:00+00:00
4   2024-01-01 04:00:00+00:00
Name: date, dtype: datetime64[us, UTC]
ENTSO-E:
datetime64[us, UTC]
0   2024-10-09 02:00:00+00:00
1   2024-10-10 11:00:00+00:00
2   2024-10-14 17:00:00+00:00
3   2024-10-15 08:00:00+00:00
4   2024-10-15 10:00:00+00:00
Name: timestamp, dtype: datetime64[us, UTC]


# Preparation for ML

# Preparing NVE time series
Will be looking at the EL3 region

In [9]:
df_el3 = (
    df[
        (df["area_type"] == "EL") &
        (df["area_id"] == 3)
    ]
    .sort_values("date")
    .copy()
)

display(df_el3[["date", "fill_level"]].head())
display(df_el3[["date", "fill_level"]].tail())

,date,fill_level
4415,1995-01-08,0.602376
4282,1995-01-15,0.569943
3985,1995-01-22,0.542203
4013,1995-01-29,0.508148
4713,1995-02-05,0.473883


,date,fill_level
8120,2026-10-18,0.724940
8134,2026-10-25,0.717647
8133,2026-11-01,0.720149
8313,2026-11-08,0.718705
8308,2026-11-15,0.697278


# Create lagged variables

In [10]:
df_el3 = df_el3.sort_values("date").copy()

df_el3["lag_1"] = df_el3["fill_level"].shift(1)
df_el3["lag_2"] = df_el3["fill_level"].shift(2)
df_el3["lag_4"] = df_el3["fill_level"].shift(4)
df_el3["lag_52"] = df_el3["fill_level"].shift(52)

display(
    df_el3[
        ["date", "fill_level", "lag_1", "lag_2", "lag_4", "lag_52"]
    ].tail(10)
)

,date,fill_level,lag_1,lag_2,lag_4,lag_52
7642,2026-09-13,0.652494,0.632579,0.627002,0.629866,0.891090
7647,2026-09-20,0.680705,0.652494,0.632579,0.637280,0.893444
8112,2026-09-27,0.715288,0.680705,0.652494,0.627002,0.899940
8117,2026-10-04,0.718754,0.715288,0.680705,0.632579,0.893539
8123,2026-10-11,0.725240,0.718754,0.715288,0.652494,0.926998
8120,2026-10-18,0.724940,0.725240,0.718754,0.680705,0.937354
8134,2026-10-25,0.717647,0.724940,0.725240,0.715288,0.920749
8133,2026-11-01,0.720149,0.717647,0.724940,0.718754,0.895342
8313,2026-11-08,0.718705,0.720149,0.717647,0.725240,0.888638
8308,2026-11-15,0.697278,0.718705,0.720149,0.724940,0.883262


Decided to add lag_52 aswell, as a candidate seasonal feature since the fill_level should follow an annual cycle.The first 52 observations were removed because the lagged features require up to 52 weeks of historical data. Rather than backfilling missing values, which could introduce future information into earlier observations, these rows were dropped. Since the dataset contains approximately 30 years of weekly observations, removing the first year represents only a small reduction in the available training data.

In [11]:
lag_columns = ["lag_1", "lag_2", "lag_4", "lag_52"]

df_model = df_el3.dropna(
    subset=["fill_level"] + lag_columns
).copy()

print("Observations available:", len(df_model))

display(df_model[["date", "fill_level"] + lag_columns].head())

Observations available: 1611


,date,fill_level,lag_1,lag_2,lag_4,lag_52
4011,1996-01-07,0.670686,0.714104,0.753256,0.817361,0.602376
4591,1996-01-14,0.630355,0.670686,0.714104,0.791204,0.569943
4027,1996-01-21,0.590421,0.630355,0.670686,0.753256,0.542203
4000,1996-01-28,0.545880,0.590421,0.630355,0.714104,0.508148
823,1996-02-04,0.502266,0.545880,0.590421,0.670686,0.473883


# Set up River Model

In [12]:
model = (
    preprocessing.StandardScaler()
    | linear_model.LinearRegression(
        optimizer=optim.SGD(0.01)
    )
)

# Train-test split
Separate it in chronological order, if not the model would learn from future periods.

In [13]:
features = ["lag_1", "lag_2", "lag_4", "lag_52"]

split = int(len(df_model) * 0.8)

train = df_model.iloc[:split]
test = df_model.iloc[split:]

print("Training:", train["date"].min(), "to", train["date"].max())
print("Testing:", test["date"].min(), "to", test["date"].max())

Training: 1996-01-07 00:00:00 to 2020-09-06 00:00:00
Testing: 2020-09-13 00:00:00 to 2026-11-15 00:00:00


# Training the model

In [14]:
for _, row in train.iterrows():
    x = row[features].to_dict()
    y = float(row["fill_level"])

    model.learn_one(x, y)

# Evaluate

In [15]:
mae = metrics.MAE()
predictions = []

for _, row in test.iterrows():
    x = row[features].to_dict()
    y = float(row["fill_level"])

    y_pred = model.predict_one(x)

    predictions.append(y_pred)
    mae.update(y, y_pred)

print(mae)

MAE: 0.028458


# Visualization

In [16]:
fig = go.Figure()

fig.add_trace(go.Scatter(
    x=test["date"],
    y=test["fill_level"],
    name="Actual"
))

fig.add_trace(go.Scatter(
    x=test["date"],
    y=predictions,
    name="Predicted"
))

fig.update_layout(
    title="River: Actual vs Predicted Reservoir Filling",
    xaxis_title="Date",
    yaxis_title="Filling level"
)

fig.show()

In [17]:
from sklearn.metrics import mean_absolute_error

baseline_mae = mean_absolute_error(
    test["fill_level"],
    test["lag_1"]
)

river_mae = mean_absolute_error(
    test["fill_level"],
    predictions
)

print(f"Baseline MAE: {baseline_mae:.4f}")
print(f"River MAE:    {river_mae:.4f}")

Baseline MAE: 0.0265
River MAE:    0.0285


# Recursive forecasting function

In [29]:
def recursive_forecast(model, history, lags, horizon=12):

    history = list(history)
    predictions = []

    for step in range(horizon):

        # Construct features from the selected lags
        x = {
            f"lag_{lag}": history[-lag]
            for lag in lags
        }

        # Predict the next week's filling level
        prediction = model.predict_one(x)

        predictions.append(prediction)

        # Add prediction to history for subsequent forecasts
        history.append(prediction)

    return predictions

# Test on historical data first

In [19]:
horizon = 12

# Actual filling levels available before the test period
history = train["fill_level"].tolist()

forecast = recursive_forecast(
    model,
    history,
    horizon=horizon
)

actual = test["fill_level"].iloc[:horizon].to_numpy()
dates = test["date"].iloc[:horizon]

print("12-week forecast MAE:", mean_absolute_error(actual, forecast))

12-week forecast MAE: 0.09133979897137973


# Check against baseline

In [20]:
baseline = [history[-1]] * horizon

print("River MAE:", mean_absolute_error(actual, forecast))
print("Baseline MAE:", mean_absolute_error(actual, baseline))

River MAE: 0.09133979897137973
Baseline MAE: 0.017846696666666644


In [25]:
import plotly.graph_objects as go

fig = go.Figure()

fig.add_trace(go.Scatter(
    x=dates,
    y=actual,
    mode="lines+markers",
    name="Actual"
))

fig.add_trace(go.Scatter(
    x=dates,
    y=forecast,
    mode="lines+markers",
    name="River forecast",
    line=dict(color="orange")
))

fig.add_trace(go.Scatter(
    x=dates,
    y=baseline,
    mode="lines",
    name="Persistence baseline",
    line=dict(dash="dash")
))

fig.update_layout(
    title="12-week Recursive Reservoir Filling Forecast",
    xaxis_title="Date",
    yaxis_title="Filling level",
    yaxis_range=[0, 1]
)

fig.show()

# Chronological backtesting

In [26]:
horizon = 12

river_errors = []
baseline_errors = []
all_values = df_model["fill_level"].to_numpy()
test_start = len(train)

for start in range(test_start, len(df_model) - horizon + 1, horizon):

    history = all_values[:start]
    actual = all_values[start:start + horizon]

    river_forecast = recursive_forecast(
        model,
        history,
        horizon=horizon
    )

    baseline_forecast = [history[-1]] * horizon

    river_errors.append(
        mean_absolute_error(actual, river_forecast)
    )

    baseline_errors.append(
        mean_absolute_error(actual, baseline_forecast)
    )

print("Average River MAE:", sum(river_errors) / len(river_errors))
print("Average baseline MAE:", sum(baseline_errors) / len(baseline_errors))

Average River MAE: 0.11701155886398316
Average baseline MAE: 0.14387118705128205


From what we see so far, the River performs better than the baseline when we introduced chronological backtesting. This means that, probably, river is better at predicting periods with larger changes, while baseline generally predicts better when the fill_level is stable. We will now experiment with different lags to see if lag_52 actually improves the model, or if we only need the recent history. 

In [27]:
def train_river(train_data, features, learning_rate=0.01):

    model = (
        preprocessing.StandardScaler()
        | linear_model.LinearRegression(
            optimizer=optim.SGD(learning_rate)
        )
    )

    for _, row in train_data.iterrows():
        x = row[features].to_dict()
        y = float(row["fill_level"])

        model.learn_one(x, y)

    return model

In [30]:
lags_a = [1, 2, 4]
lags_b = [1, 2, 4, 52]

model_a = train_river(
    train,
    features=[f"lag_{lag}" for lag in lags_a]
)

model_b = train_river(
    train,
    features=[f"lag_{lag}" for lag in lags_b]
)

In [31]:
horizon = 12
all_values = df_model["fill_level"].to_numpy()
test_start = len(train)

errors_a = []
errors_b = []

for start in range(test_start, len(df_model) - horizon + 1, horizon):

    history = all_values[:start]
    actual = all_values[start:start + horizon]

    forecast_a = recursive_forecast(
        model_a, history, lags=lags_a, horizon=horizon
    )

    forecast_b = recursive_forecast(
        model_b, history, lags=lags_b, horizon=horizon
    )

    errors_a.append(mean_absolute_error(actual, forecast_a))
    errors_b.append(mean_absolute_error(actual, forecast_b))

print(f"Without seasonal lag: {sum(errors_a)/len(errors_a):.4f}")
print(f"With seasonal lag:    {sum(errors_b)/len(errors_b):.4f}")

Without seasonal lag: 0.1259
With seasonal lag:    0.1170


Suggests that including seasonal lag improves the model. Our next step is to test different history lengths to see if it improves the model.

In [32]:
lags = [1, 2, 4, 52]
features = [f"lag_{lag}" for lag in lags]

history_lengths = [5, 10, 20, "All"]
results = {}

for years in history_lengths:

    if years == "All":
        train_subset = train
    else:
        # Approximate number of weekly observations
        n_weeks = years * 52
        train_subset = train.iloc[-n_weeks:]

    # Train a new model for each history length
    current_model = train_river(
        train_subset,
        features=features
    )

    errors = []

    for start in range(test_start, len(df_model) - horizon + 1, horizon):

        history = all_values[:start]
        actual = all_values[start:start + horizon]

        forecast = recursive_forecast(
            current_model,
            history,
            lags=lags,
            horizon=horizon
        )

        errors.append(mean_absolute_error(actual, forecast))

    results[years] = sum(errors) / len(errors)

for years, mae in results.items():
    print(f"{years} years: MAE = {mae:.4f}")

5 years: MAE = 0.1588
10 years: MAE = 0.1145
20 years: MAE = 0.1187
All years: MAE = 0.1170


Accordint to this, a history length of 10 years is preferred, which beats both 20 years and all years by a small margin. We see that 5 years has MAE = 0.159, which tells us that it may not provide enough data. Considering that 10 years is actually better than 20 years and all years, it looks like older observations doesnt really help the model. 

Our next step would be to compare forecasting horizons. 

In [33]:
lags = [1, 2, 4, 52]
features = [f"lag_{lag}" for lag in lags]

# Train on the most recent 10 years before the test period
model_10y = train_river(
    train.iloc[-10 * 52:],
    features=features
)

horizons = [4, 8, 12, 16]

for h in horizons:
    river_errors = []
    baseline_errors = []

    for start in range(test_start, len(df_model) - h + 1, h):

        history = all_values[:start]
        actual = all_values[start:start + h]

        forecast = recursive_forecast(
            model_10y, history, lags=lags, horizon=h
        )

        baseline = [history[-1]] * h

        river_errors.append(mean_absolute_error(actual, forecast))
        baseline_errors.append(mean_absolute_error(actual, baseline))

    print(
        f"{h:2d} weeks | "
        f"River MAE: {sum(river_errors)/len(river_errors):.4f} | "
        f"Baseline MAE: {sum(baseline_errors)/len(baseline_errors):.4f}"
    )

 4 weeks | River MAE: 0.0647 | Baseline MAE: 0.0620
 8 weeks | River MAE: 0.0968 | Baseline MAE: 0.1024
12 weeks | River MAE: 0.1145 | Baseline MAE: 0.1439
16 weeks | River MAE: 0.1290 | Baseline MAE: 0.1741


We see that the error increasing as length of prediction increases, which is to be expected. We also see that the baseline becomes worse and worse compared to River, which is a good sign that River can make somewhat reasonable predictions.

In [34]:
from pathlib import Path

weather_dir = Path("data/weather")

for file in weather_dir.glob("*.csv"):
    print(file.name, round(file.stat().st_size / 1_000_000, 2), "MB")

NO 3_2024-01-01_2024-12-31_average.csv 0.45 MB


# Connecting with Open-Meteo
My next goal is to introduce open-meteo data, staarting with weekly precipitation and temperature. These factors should increase the accuracy of the model.
I will do this by first using only 100 locations instead of 1000, and fetch daily, not hourly observations from open-meteo. This should make the downloading quicker, and we could of course expand the locations later again if that is feasible.

In [53]:
coordinates_path = Path("data/weather/zone_coordinates.json")

with open(coordinates_path, "r", encoding="utf-8") as f:
    zone_coordinates = json.load(f)

no3_coordinates = zone_coordinates["NO 3"][:100]

print(f"Selected {len(no3_coordinates)} locations")
print(no3_coordinates[:5])

Selected 100 locations
[[9.242168144376933, 63.747286169983724], [10.634300385849494, 63.02200095168361], [9.658707025216366, 63.104202632143185], [6.939771425111094, 62.1563555293372], [7.497796087593908, 62.438419936067476]]


# Testing for 1 coordinate

In [54]:
# Coordinates are stored as [longitude, latitude]
lon, lat = no3_coordinates[0]

url = "https://archive-api.open-meteo.com/v1/archive"

params = {
    "latitude": lat,
    "longitude": lon,
    "start_date": "2024-01-01",
    "end_date": "2024-12-31",
    "daily": "temperature_2m_mean,precipitation_sum",
    "timezone": "Europe/Oslo"
}

response = requests.get(url, params=params, timeout=60)
response.raise_for_status()

data = response.json()

df_weather_test = pd.DataFrame(data["daily"])
df_weather_test["time"] = pd.to_datetime(df_weather_test["time"])

print(df_weather_test.head())
print(df_weather_test.shape)

        time  temperature_2m_mean  precipitation_sum
0 2024-01-01                 -0.9                0.0
1 2024-01-02                 -3.1                0.0
2 2024-01-03                 -4.7                0.0
3 2024-01-04                 -3.6                0.1
4 2024-01-05                  0.5                1.6
(366, 3)


# Function for 

# AI Usage

# Log

Timezone investigation
We investigated the timestamp formats of the four data sources used in the project.
- NVE Magasinstatistikk: Weekly observations represented by timezone-naive dates. These identify reporting periods rather than exact moments.
- Norges Bank: Daily exchange-rate observations represented by timezone-naive dates. No timezone conversion is necessary for the date itself.
- Open-Meteo: Hourly weather observations explicitly represented in UTC (datetime64[us, UTC]).
- ENTSO-E: Electricity-transfer observations stored as timezone-naive timestamps (datetime64[us]), although the API uses UTC. The timezone information should be restored after verifying that no local-time conversion occurred during processing.
For the forecasting model, we plan to aggregate the data into weekly observations corresponding to the NVE reporting periods. Hourly data will be kept in UTC until aggregation, while daily and weekly reporting dates will retain their calendar-date meaning.

Care must be taken when aligning data sources to avoid daylight saving errors and introducing information that was not available at the time of prediction.

ENTSOE: UTC timestamps from the XML but returned as timezone naive by MongoDB.

The four datasets have different temporal resolutions and timezone representations. NVE provides weekly reservoir filling levels, Norges Bank daily exchange rates, Open-Meteo hourly weather observations, and ENTSO-E electricity transfers at 15-minute or hourly intervals. Open-Meteo and ENTSO-E use UTC, while NVE and Norges Bank use reporting dates without explicit timezones.

We decided to aggregate the explanatory variables to weekly resolution, aligned with the NVE observations. This reduces computational requirements and provides consistent input data for the forecasting model. Temperature and exchange rates will be averaged, while precipitation and electricity transfers will be summed over each week. Missing observations and data availability at prediction time must be considered.

The initial River linear regression model achieved an MAE of 0.028, compared with 0.0265 for a persistence baseline. The baseline therefore slightly outperformed River in one-week-ahead predictions. This suggests that recent reservoir filling levels are highly informative and that the additional model complexity did not improve short-term accuracy. Further evaluation will focus on multi-week forecasting, where seasonal patterns and additional explanatory variables may become more important.

### Recursive forecasting: River vs. persistence

We evaluated 12-week recursive forecasts across multiple non-overlapping periods in the test dataset.

The River linear regression model achieved an average MAE of 0.1170, compared with 0.1439 for the persistence baseline, representing approximately an 18.7% reduction in error.

In the initial September–November 2020 forecast, persistence performed considerably better because reservoir filling levels remained relatively stable. However, across multiple periods, River achieved better average performance.

A possible explanation is that the lagged variables allow River to capture historical trends and seasonal patterns, whereas persistence assumes the filling level remains constant throughout the forecast horizon.

Nevertheless, the error remains relatively high, indicating potential for improvement through additional explanatory variables, alternative model parameters and further testing.